# 01 — Data inventory and import

**Objective.** Inventory the publishable and restricted inputs and inspect their schemas.

**Business relevance.** Establish a transparent descriptive evidence base without merging incompatible populations or implying causality.

**Sources and dataset IDs.** Destatis 12411-0005, 12411-0013 and 23631-0001; RKI/GEDA obesity trend; WIdO PharMaAnalyst A10BJ; selected EMA EPAR records.  
**Acquisition.** Destatis flat files downloaded through GENESIS table interfaces; official RKI publication download; manual WIdO portal export obtained through legitimate access; manual consultation of individual EMA EPAR pages. No acquisition API or OCR is implemented.  
**Exact inputs.** `data/raw/destatis/*.zip`, `data/raw/rki/JHealthMonit_2025_01_Adipositas_Rauchen.pdf`, public CSVs in `data/processed/`, and optional restricted `data/external/wirkst_export.csv`.  
**Original formats.** Destatis ZIP/semicolon-delimited CSV; RKI PDF; WIdO semicolon-delimited Windows-1252 CSV; EMA web/HTML and regulatory product information.  
**Transformation status.** This notebook inventories files, inspects Destatis archive members and previews processed tables. It does not build the processed tables from RAW. RKI values were manually transcribed from published aggregates; EMA fields were manually curated.  
**Outputs.** Displayed inventory, archive-member and schema previews only; no persisted analytical file.  
**Reproduction limitation.** Public RAW and processed snapshots are auditable, but Destatis RAW-to-processed reconstruction is not fully automated. WIdO requires an independently obtained legitimate export and its row-level data are not redistributed.  
**Period / units.** 2003/04–2025, source dependent; source-specific units.

Last updated: 2026-08-27

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed'
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
files=[]
for p in sorted((ROOT/'data').rglob('*')):
    if p.is_file(): files.append({'path':p.relative_to(ROOT).as_posix(),'format':p.suffix.lower(),'bytes':p.stat().st_size})
inventory=pd.DataFrame(files)
display(inventory)

,path,format,bytes
0,data/external/wido_expected_schema.csv,.csv,145
1,data/powerbi_exports/README.md,.md,459
2,data/processed/ema_glp1_eu_authorisations.csv,.csv,1445
3,data/processed/fact_disease_cost_observed.csv,.csv,1659
4,data/processed/fact_obesity_observed.csv,.csv,2471
5,data/processed/fact_population_observed.csv,.csv,75122
6,data/processed/fact_population_state_age_sex.csv,.csv,2479061
7,data/raw/destatis/12411-0005_de_flat.zip,.zip,4722
8,data/raw/destatis/12411-0013_de_flat.zip,.zip,230420
9,data/raw/destatis/23631-0001_de_flat.zip,.zip,7180


## Flow map

Official files → explicit import → schema and quality checks → processed analytical DataFrames → notebooks and Matplotlib → CSV interface → Power BI and DAX reconciliation. WIdO is external and restricted.

In [3]:
import zipfile
for archive in sorted((ROOT/'data/raw/destatis').glob('*.zip')):
    with zipfile.ZipFile(archive) as zf:
        print(archive.name, 'contains', zf.namelist())

for name in ['fact_population_observed.csv','fact_obesity_observed.csv','fact_disease_cost_observed.csv','ema_glp1_eu_authorisations.csv']:
    df=pd.read_csv(DATA/name)
    print(name, df.shape, list(df.columns))
    display(df.head(2))

12411-0005_de_flat.zip contains ['12411-0005_de_flat.csv']
12411-0013_de_flat.zip contains ['12411-0013_de_flat.csv']
23631-0001_de_flat.zip contains ['23631-0001_de_flat.csv']
fact_population_observed.csv (435, 12) ['date_key', 'reference_date', 'year', 'age_code', 'age_label', 'population_persons', 'unit', 'source_id', 'input_class', 'geography', 'denominator', 'series_basis']


,date_key,reference_date,year,age_code,age_label,population_persons,unit,source_id,input_class,geography,denominator,series_basis
0,2021,2021-12-31,2021,ALT000,unter 1 Jahr,791254,Anzahl,DESTATIS-12411-0005,official population estimate,Germany,registered resident population at year end,Census 2011 basis
1,2021,2021-12-31,2021,ALT001,1-Jährige,780795,Anzahl,DESTATIS-12411-0005,official population estimate,Germany,registered resident population at year end,Census 2011 basis


fact_obesity_observed.csv (9, 15) ['date_key', 'period_start_year', 'period_end_year', 'estimate_pct', 'ci95_lower_pct', 'ci95_upper_pct', 'indicator', 'population', 'stratum', 'unit', 'source_id', 'input_class', 'geography', 'denominator', 'method_note']


,date_key,period_start_year,period_end_year,estimate_pct,ci95_lower_pct,ci95_upper_pct,indicator,population,stratum,unit,source_id,input_class,geography,denominator,method_note
0,2003/2004,2003,2004,12.2,11.5,12.9,Obesity prevalence (BMI >= 30 kg/m2),Adults aged 18+ in Germany,Total,percent,RKI-GEDA-TREND-2025,official survey estimate,Germany,weighted adult survey population,Self-reported height and weight; directly age-...
1,2006,2006,2006,13.7,12.5,15.0,Obesity prevalence (BMI >= 30 kg/m2),Adults aged 18+ in Germany,Total,percent,RKI-GEDA-TREND-2025,official survey estimate,Germany,weighted adult survey population,Self-reported height and weight; directly age-...


fact_disease_cost_observed.csv (8, 14) ['date_key', 'year', 'diagnosis_code', 'diagnosis_label', 'metric_code', 'metric', 'value', 'unit', 'quality_flag', 'source_id', 'input_class', 'geography', 'payer_scope', 'denominator']


,date_key,year,diagnosis_code,diagnosis_label,metric_code,metric,value,unit,quality_flag,source_id,input_class,geography,payer_scope,denominator
0,2020,2020,E10-E14,Diabetes mellitus,KOS025,Krankheitskosten,7681.0,Mill. EUR,e,DESTATIS-23631-0001,official disease-cost estimate,Germany,all payers,national total across all payers
1,2020,2020,E10-E14,Diabetes mellitus,KOS029,Krankheitskosten je Einwohner,90.0,EUR,e,DESTATIS-23631-0001,official disease-cost estimate,Germany,all payers,resident population


ema_glp1_eu_authorisations.csv (6, 9) ['active_ingredient', 'reference_product', 'authorisation_date', 'authorisation_sort', 'indication', 'ema_url', 'authorisation_scope', 'accessed_on', 'input_class']


,active_ingredient,reference_product,authorisation_date,authorisation_sort,indication,ema_url,authorisation_scope,accessed_on,input_class
0,Exenatid,Byetta,20.11.2006,20061120,Typ-2-Diabetes,https://www.ema.europa.eu/en/medicines/human/E...,EU-weite Zulassung; kein Datum der Markteinfüh...,2026-08-24,observed_regulatory_data
1,Liraglutid,Victoza,30.06.2009,20090630,Typ-2-Diabetes,https://www.ema.europa.eu/en/medicines/human/E...,EU-weite Zulassung; kein Datum der Markteinfüh...,2026-08-24,observed_regulatory_data


## Limitations, outputs and conclusion

The analysis preserves source periods, units and denominators; it does not impute, interpolate or claim causality. Outputs are the displayed summary tables and Matplotlib figures. The conclusion is descriptive and should be read with the source-specific limitations above.